# Notebook 01 — GEE Data Preprocessing
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Pull and export all satellite datasets from Google Earth Engine for use in Colab notebooks.  
**Output:** Preprocessed rasters exported to Google Drive at `Kitui_WASHLAB/satellite/`.

---
## Datasets pulled
| Dataset | GEE collection | Resolution | Use in analysis |
|---------|---------------|------------|------------------|
| NDVI | MODIS MOD13A3 | 1km monthly | Vegetation condition (WASI C3) |
| NDVI baseline | MODIS MOD13A3 1995–2005 | 1km | Anomaly reference for WASI C3 |
| Rainfall | CHIRPS v2.0 | ~5km pentadal | Seasonal water availability (WASI C2) |
| Land Surface Temp | MODIS MOD11A2 | 1km 8-day | Heat and aridity (diagnostic) |
| Evapotranspiration | MODIS MOD16A2 | 500m 8-day | Water consumption (diagnostic) |
| Terrain / DEM | SRTM 30m | 30m | Slope and accessibility (WASI C5) |
| Population density | WorldPop 2020 | 100m | Exposure weighting (WASI C4) |
| Soil moisture | ERA5-Land | ~9km monthly | Groundwater recharge proxy (diagnostic) |
| Surface water | JRC Global Surface Water | 30m | Permanent vs seasonal (Phase 1 map) |

---
## Before running
1. Sign up for Google Earth Engine: https://earthengine.google.com
2. Create a GEE Cloud project and note the project ID
3. Replace `'your-gee-project-id'` in the Setup cell below
4. Ensure your Google Drive has enough space (~2 GB for all exports)

Exports will appear in `My Drive / Kitui_WASHLAB / satellite /` when complete.
Check progress in the GEE Task Manager: https://code.earthengine.google.com/tasks

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install earthengine-api geemap geopandas -q

import ee
import geemap
import os
from google.colab import drive

drive.mount('/content/drive')

# TODO: replace with your GEE Cloud project ID
GEE_PROJECT = 'your-gee-project-id'

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

DRIVE_ROOT = '/content/drive/MyDrive/Kitui_WASHLAB/'
SAT_FOLDER = 'Kitui_WASHLAB/satellite'   # GEE Drive export folder (relative to My Drive)
SAT_LOCAL  = DRIVE_ROOT + 'satellite/'   # Local path after export

os.makedirs(SAT_LOCAL, exist_ok=True)
print('Setup complete')
print(f'Exports will go to: My Drive / {SAT_FOLDER}')

In [ ]:
# ── 1. Define Kitui County study area ─────────────────────────────────────────
# Load Kitui boundary from FAO GAUL (Level 1 = county level in Kenya)
kenya = ee.FeatureCollection('FAO/GAUL/2015/level1')
kitui = kenya.filter(ee.Filter.eq('ADM1_NAME', 'Kitui'))
kitui_geom = kitui.geometry()

# Verify on the map
Map = geemap.Map()
Map.centerObject(kitui_geom, 8)
Map.addLayer(kitui_geom, {'color': '0B5394'}, 'Kitui County')
print('Kitui geometry loaded — check the map below')
Map

In [ ]:
# ── 2. NDVI — MODIS MOD13A3 ───────────────────────────────────────────────────
# Two products exported:
#   a) Long-term mean 2000–2025 — used as current vegetation condition
#   b) Baseline mean 1995–2005  — used as the anomaly reference in WASI C3
#      (MODIS only starts 2000 — 2000–2004 used as earliest available baseline)

ndvi_coll = (ee.ImageCollection('MODIS/061/MOD13A3')
               .filterBounds(kitui_geom)
               .select('NDVI')
               .map(lambda img: img.multiply(0.0001)           # MODIS scale factor
                                   .copyProperties(img, ['system:time_start'])))

# a) 2000–2025 mean — current vegetation condition
ndvi_mean_2000_2025 = (ndvi_coll
                         .filterDate('2000-01-01', '2025-12-31')
                         .mean()
                         .clip(kitui_geom)
                         .rename('ndvi_mean'))

# b) 2000–2004 baseline — earliest 5-year window MODIS can provide
#    Label clearly in all outputs: MODIS baseline not 1995
ndvi_baseline_2000_2004 = (ndvi_coll
                             .filterDate('2000-01-01', '2004-12-31')
                             .mean()
                             .clip(kitui_geom)
                             .rename('ndvi_baseline'))

# Annual mean series for trend analysis (optional)
years = ee.List.sequence(2000, 2025)
def annual_ndvi(year):
    y = ee.Number(year).int()
    return (ndvi_coll
              .filter(ee.Filter.calendarRange(y, y, 'year'))
              .mean()
              .set('year', y)
              .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

ndvi_annual = ee.ImageCollection(years.map(annual_ndvi))
print('NDVI collections ready')
print('  2000–2025 mean: current vegetation condition')
print('  2000–2004 baseline: anomaly reference (earliest MODIS window)')

In [ ]:
# ── 3. Rainfall — CHIRPS v2.0 ─────────────────────────────────────────────────
# CHIRPS pentadal (5-day) precipitation, 1981–2025
# Exported products:
#   - 30-year baseline annual mean (1981–2010) — denominator for deficit
#   - Short rains mean (Oct–Dec)
#   - Long rains mean (Mar–May)
#   - Recent annual total (2020–2024) — numerator for deficit

chirps = (ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
            .filterBounds(kitui_geom)
            .select('precipitation'))

# 30-year baseline annual mean (1981–2010 WMO standard period)
# Sum all pentads per year then average over 30 years
baseline_years = ee.List.sequence(1981, 2010)
def annual_rain(year):
    y = ee.Number(year).int()
    return (chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
                  .sum()
                  .set('year', y)
                  .set('system:time_start', ee.Date.fromYMD(y, 1, 1).millis()))

rainfall_baseline_30yr = (ee.ImageCollection(baseline_years.map(annual_rain))
                            .mean()
                            .clip(kitui_geom)
                            .rename('rainfall_baseline_mm_yr'))

# Recent 5-year annual total mean (2020–2024) — for deficit calculation
recent_years = ee.List.sequence(2020, 2024)
rainfall_recent_5yr = (ee.ImageCollection(recent_years.map(annual_rain))
                         .mean()
                         .clip(kitui_geom)
                         .rename('rainfall_recent_mm_yr'))

# Seasonal composites
short_rains = (chirps.filter(ee.Filter.calendarRange(10, 12, 'month'))
                     .filterDate('2000-01-01', '2025-12-31')
                     .mean()
                     .clip(kitui_geom)
                     .rename('short_rains_mean_pentad'))

long_rains  = (chirps.filter(ee.Filter.calendarRange(3, 5, 'month'))
                     .filterDate('2000-01-01', '2025-12-31')
                     .mean()
                     .clip(kitui_geom)
                     .rename('long_rains_mean_pentad'))

print('Rainfall layers ready')
print('  Baseline: 1981–2010 (WMO 30-year standard)')
print('  Recent:   2020–2024 (5-year mean)')
print('  Seasonal: short rains (Oct–Dec), long rains (Mar–May)')

In [ ]:
# ── 4. Terrain — SRTM 30m ─────────────────────────────────────────────────────
srtm  = ee.Image('USGS/SRTMGL1_003').clip(kitui_geom)
slope = ee.Terrain.slope(srtm).rename('slope_deg')
elev  = srtm.rename('elevation_m')

# Aspect (for field context — not used in WASI)
aspect = ee.Terrain.aspect(srtm).rename('aspect_deg')

print('Terrain ready')
print('  Elevation, slope, aspect from SRTM 30m (USGS/SRTMGL1_003)')

In [ ]:
# ── 5. Population density — WorldPop 2020 ─────────────────────────────────────
pop = (ee.ImageCollection('WorldPop/GP/100m/pop')
         .filter(ee.Filter.eq('country', 'KEN'))
         .filter(ee.Filter.eq('year', 2020))
         .first()
         .clip(kitui_geom)
         .rename('population_100m'))

print('Population density ready')
print('  WorldPop Kenya 2020, 100m resolution')

In [ ]:
# ── 6. JRC Global Surface Water ───────────────────────────────────────────────
jrc = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(kitui_geom)

# Seasonality: 0=no water, 1-11=seasonal, 12=permanent
water_seasonality = jrc.select('seasonality').rename('water_seasonality')
# Occurrence: % of Landsat observations when water was present (1984–2021)
water_occurrence  = jrc.select('occurrence').rename('water_occurrence_pct')
# Transition: change between epochs (stable, new, lost, seasonal, etc.)
water_transition  = jrc.select('transition').rename('water_transition')

print('JRC Surface Water ready')
print('  Seasonality, occurrence, and transition from JRC/GSW1_4')

In [ ]:
# ── 7. Land Surface Temperature — MODIS MOD11A2 ───────────────────────────────
lst = (ee.ImageCollection('MODIS/061/MOD11A2')
         .filterDate('2000-01-01', '2025-12-31')
         .filterBounds(kitui_geom)
         .select('LST_Day_1km')
         .map(lambda img: img.multiply(0.02)
                             .subtract(273.15)   # Kelvin → Celsius
                             .copyProperties(img, ['system:time_start'])))

lst_mean = lst.mean().clip(kitui_geom).rename('lst_mean_celsius')

# Seasonal LST for dry season heat stress
lst_dry = (lst.filter(ee.Filter.calendarRange(1, 3, 'month'))   # Jan–Mar dry season
              .mean().clip(kitui_geom).rename('lst_dry_season_celsius'))

print('Land Surface Temperature ready')
print('  Annual mean and dry-season (Jan–Mar) mean from MODIS MOD11A2')

In [ ]:
# ── 8. Evapotranspiration — MODIS MOD16A2 ────────────────────────────────────
et = (ee.ImageCollection('MODIS/061/MOD16A2')
        .filterDate('2000-01-01', '2025-12-31')
        .filterBounds(kitui_geom)
        .select('ET')
        .map(lambda img: img.multiply(0.1)   # scale factor: kg/m²/8day → mm/8day
                            .copyProperties(img, ['system:time_start'])))

et_mean = et.mean().clip(kitui_geom).rename('et_mean_mm_8day')

print('Evapotranspiration ready')
print('  Annual mean 8-day ET from MODIS MOD16A2 (mm/8-day)')

In [ ]:
# ── 9. Soil moisture — ERA5-Land ──────────────────────────────────────────────
# ERA5-Land volumetric soil water in layer 1 (0–7 cm depth)
# Used as a groundwater recharge proxy — not a WASI component but useful
# for field validation and seasonal water availability narrative

era5 = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
          .filterDate('2000-01-01', '2024-12-31')
          .filterBounds(kitui_geom)
          .select('volumetric_soil_water_layer_1'))

soil_moisture_mean = (era5.mean()
                         .clip(kitui_geom)
                         .rename('soil_moisture_mean_m3m3'))

# Seasonal: dry season (Jan–Mar) vs wet season (Mar–May)
soil_dry = (era5.filter(ee.Filter.calendarRange(1, 3, 'month'))
                .mean().clip(kitui_geom).rename('soil_moisture_dry_m3m3'))

soil_wet = (era5.filter(ee.Filter.calendarRange(3, 5, 'month'))
                .mean().clip(kitui_geom).rename('soil_moisture_wet_m3m3'))

print('Soil moisture ready')
print('  ERA5-Land volumetric soil water layer 1 (0–7cm), ~9km resolution')
print('  Annual mean, dry-season mean, wet-season mean')

In [ ]:
# ── 10. Export all layers to Google Drive ─────────────────────────────────────
# All rasters exported at 500m to balance resolution and file size.
# WorldPop (100m) and SRTM slope (30m) exported at native resolution.

BASE_PARAMS = {
    'region':    kitui_geom,
    'crs':       'EPSG:4326',
    'folder':    SAT_FOLDER,
    'maxPixels': 1e13,
    'fileFormat':'GeoTIFF',
}

# (image, filename, scale_m)
EXPORTS = [
    # NDVI
    (ndvi_mean_2000_2025,       'kitui_ndvi_mean_2000_2025',      500),
    (ndvi_baseline_2000_2004,   'kitui_ndvi_baseline_2000_2004',  500),
    # Rainfall
    (rainfall_baseline_30yr,    'kitui_rainfall_baseline_1981_2010', 5000),
    (rainfall_recent_5yr,       'kitui_rainfall_recent_2020_2024',   5000),
    (short_rains,               'kitui_short_rains_mean',            5000),
    (long_rains,                'kitui_long_rains_mean',             5000),
    # Terrain
    (elev,                      'kitui_elevation_srtm30',            30),
    (slope,                     'kitui_slope_deg',                   30),
    (aspect,                    'kitui_aspect_deg',                  30),
    # Population
    (pop,                       'kitui_worldpop_2020',              100),
    # Surface water
    (water_seasonality,         'kitui_jrc_water_seasonality',      30),
    (water_occurrence,          'kitui_jrc_water_occurrence',        30),
    (water_transition,          'kitui_jrc_water_transition',        30),
    # Temperature
    (lst_mean,                  'kitui_lst_mean_celsius',           1000),
    (lst_dry,                   'kitui_lst_dry_season_celsius',     1000),
    # Evapotranspiration
    (et_mean,                   'kitui_et_mean_mm_8day',             500),
    # Soil moisture
    (soil_moisture_mean,        'kitui_soil_moisture_mean',         9000),
    (soil_dry,                  'kitui_soil_moisture_dry',          9000),
    (soil_wet,                  'kitui_soil_moisture_wet',          9000),
]

tasks = []
for image, filename, scale in EXPORTS:
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=filename,
        fileNamePrefix=filename,
        scale=scale,
        **BASE_PARAMS
    )
    task.start()
    tasks.append((filename, task))
    print(f'  Started: {filename}  ({scale}m)')

print(f'\n{len(tasks)} export tasks submitted.')
print('Check progress at: https://code.earthengine.google.com/tasks')
print('Exports take 5–30 minutes each depending on resolution and area.')
print('Run Notebook 02 only after all exports show "COMPLETED" in the task manager.')

In [ ]:
# ── 11. Monitor export tasks ──────────────────────────────────────────────────
# Run this cell periodically to check task status without leaving the notebook.
# Re-run until all tasks show COMPLETED.

import time

statuses = {}
for filename, task in tasks:
    status = task.status()['state']
    statuses[filename] = status

completed  = sum(1 for s in statuses.values() if s == 'COMPLETED')
running    = sum(1 for s in statuses.values() if s == 'RUNNING')
pending    = sum(1 for s in statuses.values() if s == 'READY')
failed     = sum(1 for s in statuses.values() if s == 'FAILED')

print(f'Task status: {completed} done | {running} running | {pending} pending | {failed} failed')
print()
for filename, status in statuses.items():
    icon = '✅' if status == 'COMPLETED' else '🔄' if status == 'RUNNING' else '⏳' if status == 'READY' else '❌'
    print(f'  {icon} {filename}: {status}')

if failed > 0:
    print('\n⚠ Failed tasks — re-run the export cell for the failed files.')
if completed == len(tasks):
    print('\n✅ All exports complete — proceed to Notebook 02.')

In [ ]:
# ── 12. Quick QA visualisation ─────────────────────────────────────────────────
# Visualise key layers in the GEE map viewer to confirm exports look correct
# Run after exports complete and files appear in Drive.

Map2 = geemap.Map()
Map2.centerObject(kitui_geom, 8)

# Kitui outline
Map2.addLayer(kitui_geom, {'color': '0B5394'}, 'Kitui boundary')

# NDVI mean
Map2.addLayer(
    ndvi_mean_2000_2025,
    {'min': 0.0, 'max': 0.8, 'palette': ['#d7191c','#ffffbf','#1a9641']},
    'NDVI mean 2000–2025'
)

# Rainfall baseline
Map2.addLayer(
    rainfall_baseline_30yr,
    {'min': 200, 'max': 1200, 'palette': ['#d7191c','#ffffbf','#2c7bb6']},
    'Rainfall baseline 1981–2010 (mm/yr)'
)

# Slope
Map2.addLayer(
    slope,
    {'min': 0, 'max': 30, 'palette': ['white','#888888','black']},
    'Slope (degrees)'
)

# JRC surface water seasonality
Map2.addLayer(
    water_seasonality,
    {'min': 0, 'max': 12, 'palette': ['white','#9DC3E6','#2E75B6']},
    'Water seasonality (JRC)'
)

# Soil moisture
Map2.addLayer(
    soil_moisture_mean,
    {'min': 0.05, 'max': 0.40, 'palette': ['#d7191c','#ffffbf','#2c7bb6']},
    'Soil moisture mean (ERA5-Land)'
)

Map2.add_layer_control()
print('QA map — toggle layers to verify coverage and values')
Map2